In [1]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
 
assistant = client.beta.assistants.create(
  
  name="Medical Assistant",
  instructions="You are an expert medical analyst. Use you knowledge base to answer questions about medical research papers.",
  model="gpt-4-turbo",
  # model="gpt-3.5-turbo-instruct",
  tools=[{"type": "file_search"}],
)

In [2]:
# Create a vector store caled "Medical Documents"
vector_store = client.beta.vector_stores.create(name="Medical Documents")
 
# Ready the files for upload to OpenAI
file_paths = ["documents/13-0652.pdf", "documents/Blouin_2012.pdf", "documents/Guyeux_2024.pdf"]
file_streams = [open(path, "rb") for path in file_paths]
 
# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

In [3]:
# You can print the status and the file counts of the batch to see the result of this operation.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=3, failed=0, in_progress=0, total=3)


In [4]:
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

In [5]:
# Upload the user provided file to OpenAI
message_file = client.files.create(
  file=open("documents/13-0652.pdf", "rb"), purpose="assistants"
)

In [6]:
# Create a thread and attach the file to the message
thread = client.beta.threads.create(
  messages=[
    {
      "role": "user",
      "content": "Où trouve-t-on principalement des M.canettii ?",
      # Attach the new file to the message.
      "attachments": [
        { "file_id": message_file.id, "tools": [{"type": "file_search"}] }
      ],
    }
  ]
)
 
# The thread now has a vector store with that file in its tool resources.
print(thread.tool_resources.file_search)

ToolResourcesFileSearch(vector_store_ids=['vs_k61QbmMy4wvW7IM2Jh3kUz93'])


In [7]:
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id, assistant_id=assistant.id
)

messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))

message_content = messages[0].content[0].text
annotations = message_content.annotations
citations = []
for index, annotation in enumerate(annotations):
    message_content.value = message_content.value.replace(annotation.text, f"[{index}]")
    if file_citation := getattr(annotation, "file_citation", None):
        cited_file = client.files.retrieve(file_citation.file_id)
        citations.append(f"[{index}] {cited_file.filename}")

print(message_content.value)
print("\n".join(citations))

Mycobacterium canettii, souvent impliquée dans des cas de tuberculose, est généralement trouvée principalement à Djibouti et dans la Corne de l'Afrique. Il y a aussi des cas documentés dans d'autres régions, mais ils sont généralement liés à des individus qui ont voyagé ou qui ont des liens avec cette région. M. canettii est connue pour sa rareté et sa distribution géographique limitée par rapport à d'autres mycobactéries causant la tuberculose.

